<a href="https://colab.research.google.com/github/kimrichies/MATERNAL-HEALTH-RISK-PREDICTION-MACHINE-LEARNING-TRAINING-SCRIPT/blob/main/ML_Training_Mobile_HealthAps_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# In Colab, run:
!git config --global user.name "kimrichies"
!git config --global user.email "kimrichies@gmail.com"
!git config --global credential.helper store

In [ ]:
"""
================================================================================
  MATERNAL HEALTH RISK PREDICTION — MACHINE LEARNING TRAINING SCRIPT
  Mbarara University of Science and Technology (MUST)
  MSc Health Informatics — HIN 7102
  Author  : Dr. Richard Kimera
  Version : 1.0  |  2026
================================================================================

WHAT THIS SCRIPT DOES
─────────────────────
This script trains a neural network to predict maternal health risk from six
clinical vitals — Age, Systolic Blood Pressure, Diastolic Blood Pressure,
Blood Glucose, Body Temperature, and Heart Rate — and classifies each patient
into one of three risk categories:
    • Class 0 — Low Risk
    • Class 1 — Moderate Risk
    • Class 2 — High Risk

The trained model is exported as a TensorFlow Lite (.tflite) file for embedding
directly inside the Android application without any internet connection.

HOW TO RUN THIS SCRIPT
──────────────────────
1.  Make sure you have Python 3.8 or newer installed.
        Check:  python --version

2.  Install the required libraries (one-time setup):
        pip install tensorflow scikit-learn matplotlib seaborn pandas numpy

3.  Run the script from your terminal:
        python maternal_health_train.py

4.  When it finishes you will find these new files in the same folder:
        maternal_health.tflite      ← the model for Android
        scaler_params.json          ← normalisation values for Android
        plots/                      ← folder containing all graphs

READING THE OUTPUT
──────────────────
• "Epoch X/60" lines — each epoch is one full pass through the training data.
  Watch the "accuracy" number climb toward 1.0 and the "loss" fall toward 0.
• "Test accuracy" — how well the model performs on data it has never seen.
  Anything above 0.85 (85%) is good for this type of clinical screening tool.
• The confusion matrix plot shows where the model makes mistakes.
• The ROC curves show the trade-off between true positives and false positives.

GLOSSARY FOR BEGINNERS
──────────────────────
• Neural Network  : A mathematical model loosely inspired by the brain, made of
                    layers of numbers that learn patterns from examples.
• Epoch           : One complete training cycle through all training examples.
• Loss            : A number measuring how wrong the model's predictions are.
                    Lower loss = better model.
• Accuracy        : Proportion of correct predictions. 1.0 = 100% correct.
• Normalisation   : Rescaling numbers so they are all on a similar scale,
                    preventing large numbers (like blood pressure ~120) from
                    drowning out small ones (like temperature ~37).
• TFLite          : A compressed version of the model designed to run on a
                    smartphone without needing a powerful computer or internet.
================================================================================
"""

'\n================================================================================\n  MATERNAL HEALTH RISK PREDICTION — MACHINE LEARNING TRAINING SCRIPT\n  Mbarara University of Science and Technology (MUST)\n  MSc Health Informatics — HIN 7102: Applied Health Informatics & AI\n  Author  : Dr. Richard Kimera\n  Version : 1.0  |  2026\n================================================================================\n\nWHAT THIS SCRIPT DOES\n─────────────────────\nThis script trains a neural network to predict maternal health risk from six\nclinical vitals — Age, Systolic Blood Pressure, Diastolic Blood Pressure,\nBlood Glucose, Body Temperature, and Heart Rate — and classifies each patient\ninto one of three risk categories:\n    • Class 0 — Low Risk\n    • Class 1 — Moderate Risk\n    • Class 2 — High Risk\n\nThe trained model is exported as a TensorFlow Lite (.tflite) file for embedding\ndirectly inside the Android application without any internet connection.\n\nHOW TO RUN THIS SCRIP

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 0 — IMPORT LIBRARIES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Think of libraries as toolboxes. Before you can use a hammer, you pick it up.
# Each import below picks up a specific toolbox:

import os            # Interact with the file system (create folders, paths)
import json          # Save data in JSON format (human-readable text files)
import warnings      # Suppress unimportant warning messages
warnings.filterwarnings("ignore")         # Keep the terminal output clean
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" # Hide TensorFlow's internal messages

import numpy as np                        # The core maths library — arrays, stats
import pandas as pd                       # Table-like data manipulation
import matplotlib.pyplot as plt           # Plotting and graph drawing
import matplotlib.gridspec as gridspec    # Advanced subplot layout control
import seaborn as sns                     # High-level, beautiful statistical plots

from sklearn.model_selection import train_test_split   # Split data into train/test
from sklearn.preprocessing import StandardScaler       # Normalise features to mean=0
from sklearn.metrics import (
    classification_report,     # Per-class precision, recall, F1-score table
    confusion_matrix,          # Count of correct vs incorrect predictions
    roc_curve,                 # Receiver Operating Characteristic curve data
    auc,                       # Area Under the Curve — single-number model quality
    ConfusionMatrixDisplay,    # Ready-made confusion matrix visualiser
)
from sklearn.utils.class_weight import compute_class_weight  # Handle imbalanced data

import tensorflow as tf                   # The machine learning framework
from tensorflow import keras              # High-level neural network building API

# ── Reproducibility ───────────────────────────────────────────────────────────
# Setting seeds means that every time you run this script you get the same
# random numbers — and therefore the same results. This is essential for
# scientific reproducibility. Without this, two runs could give different
# accuracy values even with identical code and data.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 72)
print("  MATERNAL HEALTH RISK PREDICTION — MODEL TRAINING")
print("  Mbarara University of Science and Technology | MSc Health Informatics")
print("=" * 72)
print(f"\n  TensorFlow version : {tf.__version__}")
print(f"  NumPy version      : {np.__version__}")
print(f"  Random seed        : {SEED}  (ensures reproducible results)")
print()

  MATERNAL HEALTH RISK PREDICTION — MODEL TRAINING
  Mbarara University of Science and Technology | MSc Health Informatics

  TensorFlow version : 2.20.0
  NumPy version      : 2.0.2
  Random seed        : 42  (ensures reproducible results)



In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 — CREATE THE OUTPUT FOLDER FOR ALL PLOTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)   # Create the folder; don't crash if it exists
print(f"[STEP 1]  Output folder ready → '{PLOTS_DIR}/'")


[STEP 1]  Output folder ready → 'plots/'


In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 — GENERATE THE DATASET
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# WHY SYNTHETIC DATA?
# The UCI Maternal Health Risk dataset (Ahmed et al., 2020) contains 1,014
# real patient records from IoT sensors in rural Bangladeshi hospitals.
# We generate synthetic data based on its statistical properties so that:
#  1. Student laptops don't need a separate data file to run this script.
#  2. The class distribution is balanced (equal numbers per risk level),
#     preventing the model from ignoring the rare "high risk" class.
#  3. The generation process is fully transparent and auditable.
#
# HOW SYNTHETIC GENERATION WORKS:
# For each risk class we define realistic clinical ranges, then sample random
# values from those ranges using numpy's random number generators.
# np.random.randint(a, b) → a random whole number between a and b
# np.random.uniform(a, b) → a random decimal number between a and b

print("\n[STEP 2]  Generating synthetic maternal health dataset...")
print("          (Based on UCI Maternal Health Risk Dataset distributions)")
### Access the dataset here: https://archive.ics.uci.edu/dataset/863/maternal+health+risk

N_PER_CLASS = 400   # Number of patient records to generate per risk class
                    # Total dataset = 400 × 3 classes = 1,200 records

rows = []   # We will collect each patient's data as a list and append here

for _ in range(N_PER_CLASS):
    # ── Low Risk (Class 0) ────────────────────────────────────────────────────
    # Younger mothers, normal BP, healthy glucose, normal temperature, calm HR.
    # These ranges reflect WHO guidelines for uncomplicated pregnancy.
    rows.append([
        np.random.randint(18, 35),          # Age: 18–34 years
        np.random.randint(90, 120),         # Systolic BP: 90–119 mmHg (normal)
        np.random.randint(60, 80),          # Diastolic BP: 60–79 mmHg (normal)
        round(np.random.uniform(6.0, 7.5), 1),   # Blood Glucose: 6–7.5 mmol/L
        round(np.random.uniform(36.5, 37.5), 1), # Body Temp: 36.5–37.5 °C
        np.random.randint(60, 80),          # Heart Rate: 60–79 bpm
        0                                   # LABEL: 0 = Low Risk
    ])

for _ in range(N_PER_CLASS):
    # ── Moderate Risk (Class 1) ───────────────────────────────────────────────
    # Older mothers, pre-hypertensive BP, elevated glucose (gestational diabetes
    # territory), slightly elevated temperature, faster heart rate.
    rows.append([
        np.random.randint(28, 45),
        np.random.randint(120, 140),        # Systolic: 120–139 mmHg (elevated)
        np.random.randint(80, 90),          # Diastolic: 80–89 mmHg (elevated)
        round(np.random.uniform(7.5, 10.0), 1),
        round(np.random.uniform(37.5, 38.5), 1),
        np.random.randint(75, 95),
        1                                   # LABEL: 1 = Moderate Risk
    ])

for _ in range(N_PER_CLASS):
    # ── High Risk (Class 2) ────────────────────────────────────────────────────
    # Older mothers, hypertensive crisis range (preeclampsia territory),
    # severely elevated glucose (diabetic range), fever, tachycardia.
    rows.append([
        np.random.randint(35, 65),
        np.random.randint(140, 180),        # Systolic ≥ 140 = hypertensive crisis
        np.random.randint(90, 110),         # Diastolic ≥ 90
        round(np.random.uniform(10.0, 15.0), 1),
        round(np.random.uniform(38.5, 40.0), 1),
        np.random.randint(90, 120),
        2                                   # LABEL: 2 = High Risk
    ])

# Column names — these must match exactly what the Android app expects
FEATURE_NAMES = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BloodGlucose",
    "BodyTemp",
    "HeartRate",
]
TARGET_NAME = "RiskLevel"
CLASS_NAMES = ["Low Risk", "Moderate Risk", "High Risk"]

# Convert the list of rows into a DataFrame (think of it as a spreadsheet in Python)
df = pd.DataFrame(rows, columns=FEATURE_NAMES + [TARGET_NAME])

# Shuffle the rows so Low/Moderate/High records are not grouped together.
# random_state=SEED keeps the shuffle reproducible.
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\n          Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"          Features      : {FEATURE_NAMES}")
print(f"          Target        : {TARGET_NAME}")
print("\n          Class distribution (should be perfectly balanced):")
for cls, count in df[TARGET_NAME].value_counts().sort_index().items():
    print(f"            Class {cls} — {CLASS_NAMES[cls]:<16} : {count} records")



[STEP 2]  Generating synthetic maternal health dataset...
          (Based on UCI Maternal Health Risk Dataset distributions)

          Dataset shape : 1200 rows × 7 columns
          Features      : ['Age', 'SystolicBP', 'DiastolicBP', 'BloodGlucose', 'BodyTemp', 'HeartRate']
          Target        : RiskLevel

          Class distribution (should be perfectly balanced):
            Class 0 — Low Risk         : 400 records
            Class 1 — Moderate Risk    : 400 records
            Class 2 — High Risk        : 400 records


In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3 — EXPLORATORY DATA ANALYSIS (EDA)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Before training any model, a good data scientist always looks at the data.
# EDA helps us spot anomalies, understand distributions, and see which features
# separate the risk classes most clearly.

print("\n[STEP 3]  Exploratory Data Analysis...")
print("\n          Basic statistics per feature:")
print(df[FEATURE_NAMES].describe().round(2).to_string())

# ── PLOT 1: Feature Distributions by Risk Class ───────────────────────────────
# A violin plot combines a box plot (showing median and quartiles) with a kernel
# density estimate (showing the shape of the distribution). Wider sections mean
# more data points at that value.
print("\n          Saving Plot 1: Feature distributions by risk class...")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Feature Distributions by Maternal Risk Class\n"
    "Mbarara University of Science and Technology — MSc Health Informatics",
    fontsize=14, fontweight="bold", y=1.01
)

palette = {0: "#2E7D32", 1: "#F57C00", 2: "#C62828"}   # Green / Amber / Red
label_map = {0: "Low Risk", 1: "Moderate Risk", 2: "High Risk"}
df["RiskLabel"] = df[TARGET_NAME].map(label_map)
palette_labels = {"Low Risk": "#2E7D32", "Moderate Risk": "#F57C00", "High Risk": "#C62828"}

for ax, feature in zip(axes.flat, FEATURE_NAMES):
    sns.violinplot(
        data=df,
        x="RiskLabel",
        y=feature,
        palette=palette_labels,
        order=["Low Risk", "Moderate Risk", "High Risk"],
        ax=ax,
        inner="box",      # Show a mini box-plot inside the violin
        linewidth=1.2,
    )
    ax.set_title(feature, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(feature)
    ax.tick_params(axis="x", labelsize=9)
    # Draw a light horizontal grid to make values easier to read
    ax.yaxis.grid(True, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/01_feature_distributions.png")

# ── PLOT 2: Correlation Heatmap ───────────────────────────────────────────────
# A correlation matrix shows how strongly each pair of features moves together.
# Values close to +1.0 = strong positive correlation (both go up together).
# Values close to -1.0 = strong negative correlation (one goes up, other goes down).
# Values near 0 = no linear relationship.
# This helps us check: are any features redundant (correlated with each other)?
print("          Saving Plot 2: Feature correlation heatmap...")

fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = df[FEATURE_NAMES + [TARGET_NAME]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Show only lower triangle

sns.heatmap(
    corr_matrix,
    annot=True,          # Print the correlation number inside each cell
    fmt=".2f",           # Format to 2 decimal places
    cmap="RdYlGn",       # Red = negative, Green = positive
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.8},
    mask=mask,
)
ax.set_title(
    "Feature Correlation Matrix\n(Values near ±1.0 indicate strong linear relationship)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/02_correlation_heatmap.png")

# ── PLOT 3: Class Balance Bar Chart ───────────────────────────────────────────
print("          Saving Plot 3: Class balance chart...")

fig, ax = plt.subplots(figsize=(7, 5))
counts = df[TARGET_NAME].value_counts().sort_index()
bars = ax.bar(
    [CLASS_NAMES[i] for i in counts.index],
    counts.values,
    color=["#2E7D32", "#F57C00", "#C62828"],
    edgecolor="white", linewidth=1.5, width=0.55
)
# Label each bar with its count
for bar, count in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 8,
        str(count),
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )
ax.set_title("Class Distribution — Maternal Risk Labels\n(Balanced dataset: equal records per class)", fontweight="bold")
ax.set_ylabel("Number of Patient Records")
ax.set_ylim(0, max(counts.values) * 1.2)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_class_balance.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/03_class_balance.png")


[STEP 3]  Exploratory Data Analysis...

          Basic statistics per feature:
           Age  SystolicBP  DiastolicBP  BloodGlucose  BodyTemp  HeartRate
count  1200.00     1200.00      1200.00       1200.00   1200.00    1200.00
mean     37.34      131.19        84.71          9.36     38.09      86.13
std      11.88       24.32        13.10          2.60      0.98      16.24
min      18.00       90.00        60.00          6.00     36.50      60.00
25%      29.00      112.00        74.75          7.10     37.20      74.00
50%      36.00      130.00        84.00          8.85     38.00      84.00
75%      44.00      149.00        94.25         11.30     38.80      98.00
max      64.00      179.00       109.00         15.00     40.00     119.00

          Saving Plot 1: Feature distributions by risk class...
          → Saved: plots/01_feature_distributions.png
          Saving Plot 2: Feature correlation heatmap...
          → Saved: plots/02_correlation_heatmap.png
          Saving 

In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4 — PREPARE DATA FOR TRAINING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# We split data into:
#   X  (features / inputs)  — the clinical measurements the model reads
#   y  (target / label)     — the risk level the model tries to predict

print("\n[STEP 4]  Preparing data for training...")

X = df[FEATURE_NAMES].values.astype(np.float32)   # Shape: (1200, 6)
y = df[TARGET_NAME].values                         # Shape: (1200,) — integers 0/1/2

# ── Train / Validation / Test Split ───────────────────────────────────────────
#
# We divide the data into three independent sets:
#
#   Training set (70%) — the model LEARNS from this data.
#   Validation set (10%) — used DURING training to monitor overfitting.
#   Test set (20%) — used ONLY ONCE at the very end to measure final performance.
#                    The model never sees this data during training.
#
# The key rule: NEVER use the test set for any decision during training.
# Using it would give you an over-optimistic accuracy estimate.
#
# stratify=y ensures that each split has the same class proportions.

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,        # 20% for final testing
    random_state=SEED,
    stratify=y             # Keep class proportions equal in each split
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,       # 0.125 × 80% = 10% of total dataset
    random_state=SEED,
    stratify=y_temp
)

print(f"\n          Data splits:")
print(f"            Training   : {X_train.shape[0]:>4} records ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"            Validation : {X_val.shape[0]:>4} records ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"            Test       : {X_test.shape[0]:>4} records ({X_test.shape[0]/len(X)*100:.0f}%)")

# ── Feature Normalisation (StandardScaler) ─────────────────────────────────────
#
# WHY NORMALISE?
# Our six features have very different numeric ranges:
#   Age ≈ 18–65, SystolicBP ≈ 90–180, BloodGlucose ≈ 6–15, BodyTemp ≈ 36–40
#
# Without normalisation, the neural network's weight updates would be dominated
# by features with large absolute values (like BP), and features with small
# ranges (like temperature) would barely influence the model.
#
# StandardScaler transforms each feature to have:
#   mean (μ) = 0      (centre the values around zero)
#   std dev (σ) = 1   (all features have the same spread)
#
# Formula applied to every value x: z = (x − μ) / σ
#
# CRITICAL RULE: Fit the scaler ONLY on the training data.
# Then apply the SAME scaler to validation and test data.
# Fitting on all data would "leak" test information into training — data leakage.

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)    # Compute μ and σ, then scale
X_val_s   = scaler.transform(X_val)          # Use training μ and σ — no refit
X_test_s  = scaler.transform(X_test)         # Same

print(f"\n          StandardScaler fitted on training data:")
for i, fname in enumerate(FEATURE_NAMES):
    print(f"            {fname:<15} mean={scaler.mean_[i]:>8.3f}   std={scaler.scale_[i]:>7.3f}")

# ── Save scaler parameters ─────────────────────────────────────────────────────
# The Android app must apply exactly the same normalisation to new inputs.
# We save μ and σ for each feature so they can be hard-coded in Java.
scaler_params = {
    "mean":          scaler.mean_.tolist(),
    "scale":         scaler.scale_.tolist(),
    "feature_names": FEATURE_NAMES,
    "note":          "Apply: z = (x - mean[i]) / scale[i] for each feature i"
}
with open("scaler_params.json", "w") as f:
    json.dump(scaler_params, f, indent=2)
print(f"\n          Scaler parameters saved → scaler_params.json")

# ── Class Weights ──────────────────────────────────────────────────────────────
# Even with a balanced dataset, it is good practice to compute class weights.
# In a real clinical dataset, high-risk cases are rare, so we penalise the model
# more heavily for missing them. The compute_class_weight function does this:
# weight[c] = total_samples / (n_classes × samples_in_class_c)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}
print(f"\n          Class weights for training: {class_weight_dict}")



[STEP 4]  Preparing data for training...

          Data splits:
            Training   :  840 records (70%)
            Validation :  120 records (10%)
            Test       :  240 records (20%)

          StandardScaler fitted on training data:
            Age             mean=  37.288   std= 11.781
            SystolicBP      mean= 131.442   std= 24.337
            DiastolicBP     mean=  84.649   std= 13.235
            BloodGlucose    mean=   9.335   std=  2.572
            BodyTemp        mean=  38.083   std=  0.974
            HeartRate       mean=  86.255   std= 16.500

          Scaler parameters saved → scaler_params.json

          Class weights for training: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0)}


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 5 — BUILD THE NEURAL NETWORK
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# ARCHITECTURE OVERVIEW:
#
#   Input Layer   → 6 features (one neuron per feature)
#       ↓
#   Hidden Layer 1 → 32 neurons + ReLU activation + Batch Normalisation + Dropout
#       ↓
#   Hidden Layer 2 → 16 neurons + ReLU activation
#       ↓
#   Output Layer  → 3 neurons + Softmax activation (one per risk class)
#
# KEY CONCEPTS:
#
# Dense layer:
#   Every input is connected to every neuron. Each connection has a "weight"
#   (a number). During training the weights are adjusted to reduce errors.
#
# ReLU activation (Rectified Linear Unit):
#   f(x) = max(0, x)  — turns negative values into 0, keeps positive values.
#   Without activation functions, a neural network is just linear regression.
#   Activation functions let the network learn non-linear, curved boundaries.
#
# Batch Normalisation:
#   Normalises the outputs of a layer so the next layer receives stable input.
#   This makes training faster and more stable, especially in early epochs.
#
# Dropout (rate=0.3):
#   During each training step, randomly switches off 30% of neurons.
#   This sounds destructive — but it prevents overfitting.
#   Overfitting = the model memorises the training data instead of learning
#   general patterns. With dropout the network must learn redundant
#   representations, making it more robust on new, unseen data.
#
# Softmax output:
#   Converts the raw output of 3 neurons into probabilities that sum to 1.
#   E.g., [0.05, 0.15, 0.80] means: 5% Low, 15% Moderate, 80% High Risk.
#   The predicted class is whichever has the highest probability.

print("\n[STEP 5]  Building the neural network architecture...")

model = keras.Sequential(
    [
        # ── Input layer ──────────────────────────────────────────────────────
        # shape=(6,) tells Keras to expect 6 input features per patient.
        keras.layers.Input(shape=(6,), name="vitals_input"),

        # ── First hidden layer ────────────────────────────────────────────────
        keras.layers.Dense(
            32,                     # 32 neurons (units)
            activation="relu",      # ReLU activation function
            name="hidden_layer_1",
            kernel_regularizer=keras.regularizers.l2(0.001),  # L2: penalise large weights
        ),
        keras.layers.BatchNormalization(name="batch_norm_1"),
        keras.layers.Dropout(0.3, name="dropout_1"),

        # ── Second hidden layer ───────────────────────────────────────────────
        keras.layers.Dense(
            16,
            activation="relu",
            name="hidden_layer_2",
            kernel_regularizer=keras.regularizers.l2(0.001),
        ),
        keras.layers.Dropout(0.2, name="dropout_2"),

        # ── Output layer ──────────────────────────────────────────────────────
        # 3 neurons = 3 classes. Softmax converts outputs to probabilities.
        keras.layers.Dense(3, activation="softmax", name="risk_output"),
    ],
    name="MaternalRiskMLP",
)

# ── Compile the model ──────────────────────────────────────────────────────────
# Compilation links the model to its:
#   optimizer  : the algorithm that updates weights (Adam is the most popular)
#   loss       : the function that measures how wrong predictions are
#   metrics    : what to display during training (accuracy)
#
# Adam optimizer: short for Adaptive Moment Estimation. It adjusts the learning
# rate for each weight individually based on how the gradient has been changing.
# It is the default choice for most classification tasks.
#
# Sparse categorical crossentropy: the standard loss function for multi-class
# classification when labels are integers (0, 1, 2) rather than one-hot vectors.

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Print a summary table: every layer, its output shape, and parameter count
print()
model.summary()

total_params = model.count_params()
print(f"\n          Total trainable parameters: {total_params:,}")
print(f"          (Each parameter is one number the model adjusts during training)")


[STEP 5]  Building the neural network architecture...



Model: "MaternalRiskMLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 32)             │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_output (Dense)             │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 931 (3.64 KB)

 Trainable params: 867 (3.39 KB)

 Non-trainable params: 64 (256.00 B)


          Total trainable parameters: 931
          (Each parameter is one number the model adjusts during training)


In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 6 — TRAIN THE MODEL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Training Callbacks — these are automatic actions triggered during training:
#
# EarlyStopping:
#   Monitors the validation loss. If it stops improving for 12 consecutive
#   epochs, training stops automatically. This prevents overfitting.
#   restore_best_weights=True means we restore the best version of the model.
#
# ReduceLROnPlateau:
#   If the validation loss stops improving for 6 epochs, the learning rate
#   is halved. A smaller learning rate makes the model take smaller steps,
#   which can help it fine-tune when it is close to the optimal solution.

print("\n[STEP 6]  Training the model...")
print("          Reading the training log:")
print("          loss     = how wrong predictions are (lower = better)")
print("          accuracy = fraction of correct predictions (higher = better)")
print("          val_*    = same metrics on the validation set\n")

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,               # Wait 12 epochs before stopping
    restore_best_weights=True, # Keep the best model, not the last one
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,      # Multiply learning rate by 0.5
    patience=6,
    min_lr=1e-6,     # Never let learning rate go below this
    verbose=1,
)

history = model.fit(
    X_train_s, y_train,
    epochs=80,              # Maximum number of training cycles
    batch_size=32,          # Process 32 patients at a time before updating weights
    validation_data=(X_val_s, y_val),
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1,              # Print one line per epoch
)

epochs_run = len(history.history["loss"])
best_val_acc = max(history.history["val_accuracy"])
print(f"\n          Training completed in {epochs_run} epochs")
print(f"          Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")


[STEP 6]  Training the model...
          Reading the training log:
          loss     = how wrong predictions are (lower = better)
          accuracy = fraction of correct predictions (higher = better)
          val_*    = same metrics on the validation set

Epoch 1/80
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.3798 - loss: 1.6936 - val_accuracy: 0.4417 - val_loss: 1.0065 - learning_rate: 0.0010
Epoch 2/80
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6536 - loss: 0.8227 - val_accuracy: 0.7417 - val_loss: 0.7846 - learning_rate: 0.0010
Epoch 3/80
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8524 - loss: 0.4357 - val_accuracy: 0.8833 - val_loss: 0.5746 - learning_rate: 0.0010
Epoch 4/80
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9095 - loss: 0.3001 - val_accuracy: 0.9333 - val_loss: 0.4051 - learning_rate: 0.0010
Epoch 5/80
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9381 - loss: 0.2181 - val_accuracy: 0.9667 - val_loss: 0.2819 - learning_r

In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 7 — EVALUATE THE MODEL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# We now evaluate the final model on the test set — data it has NEVER seen.
# This gives us an unbiased estimate of how the model performs in the real world.

print("\n[STEP 7]  Evaluating model on unseen test data...")
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"\n          ┌─────────────────────────────────────┐")
print(f"          │  TEST LOSS     : {test_loss:.4f}              │")
print(f"          │  TEST ACCURACY : {test_acc:.4f}  ({test_acc*100:.1f}%)     │")
print(f"          └─────────────────────────────────────┘")

# ── Predictions ───────────────────────────────────────────────────────────────
# model.predict() returns probabilities: shape (n_samples, 3)
# We use argmax to get the class with the highest probability.
y_pred_proba = model.predict(X_test_s, verbose=0)   # Probability array
y_pred       = np.argmax(y_pred_proba, axis=1)       # Predicted class index

# ── Classification Report ─────────────────────────────────────────────────────
# This is the most informative summary of model performance.
# For each class it shows:
#   Precision  : Of all patients predicted as class X, what fraction really are X?
#                High precision = few false alarms.
#   Recall     : Of all actual class X patients, what fraction did we catch?
#                High recall = few missed cases. IN CLINICAL CONTEXTS, HIGH RECALL
#                FOR HIGH RISK IS THE MOST IMPORTANT METRIC.
#   F1-score   : Harmonic mean of precision and recall. Balances both.
#   Support    : How many test samples belong to this class.

print("\n          Classification Report (per-class performance):")
print("          " + "─" * 62)
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)
for line in report.split("\n"):
    print(f"          {line}")

print("\n          HOW TO INTERPRET THIS:")
print("          • Recall for 'High Risk' is the most critical value.")
print("            A low recall means high-risk mothers are being classified as")
print("            low/moderate — a potentially dangerous missed diagnosis.")
print("          • For a screening tool, prioritise recall over precision.")



[STEP 7]  Evaluating model on unseen test data...

          ┌─────────────────────────────────────┐
          │  TEST LOSS     : 0.0196              │
          │  TEST ACCURACY : 1.0000  (100.0%)     │
          └─────────────────────────────────────┘

          Classification Report (per-class performance):
          ──────────────────────────────────────────────────────────────
                         precision    recall  f1-score   support
          
               Low Risk       1.00      1.00      1.00        80
          Moderate Risk       1.00      1.00      1.00        80
              High Risk       1.00      1.00      1.00        80
          
               accuracy                           1.00       240
              macro avg       1.00      1.00      1.00       240
           weighted avg       1.00      1.00      1.00       240
          

          HOW TO INTERPRET THIS:
          • Recall for 'High Risk' is the most critical value.
            A low recall mean

In [ ]:


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 8 — GENERATE EVALUATION PLOTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n[STEP 8]  Generating evaluation plots...")

# ── PLOT 4: Training History ───────────────────────────────────────────────────
# This is the most important diagnostic plot during model development.
# It shows how loss and accuracy change over each epoch.
#
# Healthy training looks like:
#   • Training loss steadily decreases
#   • Validation loss decreases and then plateaus (levels off)
#   • Both curves remain close together
#
# Signs of overfitting:
#   • Training loss keeps falling but validation loss starts rising
#   → The model is memorising training data, not learning general patterns
#
# Signs of underfitting:
#   • Both loss values remain high and do not decrease
#   → The model is too simple or training is too short

print("          Saving Plot 4: Training history (loss + accuracy)...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Model Training History — Loss and Accuracy over Epochs\n"
    "Watch for convergence (curves flattening) and check train vs. validation gap",
    fontsize=12, fontweight="bold"
)

epochs_range = range(1, epochs_run + 1)

# Left plot: Loss
axes[0].plot(epochs_range, history.history["loss"],     label="Training Loss",   color="#1565C0", linewidth=2)
axes[0].plot(epochs_range, history.history["val_loss"], label="Validation Loss", color="#C62828", linewidth=2, linestyle="--")
axes[0].set_title("Loss per Epoch\n(lower = better)", fontsize=11)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Categorical Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)
# Mark the epoch where training stopped (best model)
best_epoch = np.argmin(history.history["val_loss"]) + 1
axes[0].axvline(best_epoch, color="green", linestyle=":", linewidth=1.5, label=f"Best epoch ({best_epoch})")
axes[0].legend()

# Right plot: Accuracy
axes[1].plot(epochs_range, history.history["accuracy"],     label="Training Accuracy",   color="#1565C0", linewidth=2)
axes[1].plot(epochs_range, history.history["val_accuracy"], label="Validation Accuracy", color="#C62828", linewidth=2, linestyle="--")
axes[1].set_title("Accuracy per Epoch\n(higher = better, max = 1.0)", fontsize=11)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (fraction correct)")
axes[1].set_ylim(0.0, 1.05)
axes[1].axhline(test_acc, color="green", linestyle=":", linewidth=1.5, label=f"Final test accuracy ({test_acc:.3f})")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/04_training_history.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/04_training_history.png")



# ── PLOT 5: Confusion Matrix ───────────────────────────────────────────────────
# A confusion matrix is a grid where:
#   • Each ROW represents the TRUE (actual) class of a patient
#   • Each COLUMN represents the PREDICTED class by the model
#   • Diagonal cells (top-left to bottom-right) = CORRECT predictions
#   • Off-diagonal cells = ERRORS
#
# Example: If row "High Risk" / column "Low Risk" has 5 — that means
# 5 actual high-risk patients were wrongly classified as low risk.
# In a clinical setting this type of error (false negative for high risk) is
# the most dangerous and should be minimised at all costs.

print("          Saving Plot 5: Confusion matrix...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Confusion Matrix — Model Predictions vs. True Labels\n"
    "Diagonal = correct predictions | Off-diagonal = errors",
    fontsize=12, fontweight="bold"
)

cm = confusion_matrix(y_test, y_pred)

# Left: raw counts
disp_counts = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp_counts.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Raw Counts", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("True Class")

# Right: normalised (shows percentages per true class = recall values)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp_norm.plot(ax=axes[1], colorbar=False, cmap="Greens", values_format=".2%")
axes[1].set_title("Normalised (% of True Class = Recall)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("True Class")

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/05_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/05_confusion_matrix.png")

# ── PLOT 6: ROC Curves (One-vs-Rest) ──────────────────────────────────────────
# A ROC (Receiver Operating Characteristic) curve shows how the model performs
# at all possible decision thresholds.
#
# The axes:
#   X-axis: False Positive Rate = FP / (FP + TN)
#           How often we incorrectly raise the alarm.
#   Y-axis: True Positive Rate (= Recall) = TP / (TP + FN)
#           How often we correctly catch a true case.
#
# A perfect model would go straight up the Y-axis and then right along the top.
# A random (useless) model gives a diagonal line from bottom-left to top-right.
#
# AUC (Area Under the Curve):
#   AUC = 1.0  → perfect model
#   AUC = 0.5  → random guessing
#   AUC > 0.85 → generally good for a clinical screening tool
#
# We compute one ROC curve per class (One-vs-Rest approach):
# For "High Risk" ROC: we ask "is this patient High Risk, or not?"

print("          Saving Plot 6: ROC curves (one-vs-rest per class)...")

fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#2E7D32", "#F57C00", "#C62828"]

# Convert integer labels to binary one-vs-rest arrays
from sklearn.preprocessing import label_binarize
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Shape: (n, 3)

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{cls_name}  (AUC = {roc_auc:.3f})")

# Diagonal reference line (= random classifier)
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random Classifier (AUC = 0.50)")

ax.set_xlabel("False Positive Rate\n(Fraction of non-cases incorrectly flagged)", fontsize=11)
ax.set_ylabel("True Positive Rate (Recall)\n(Fraction of true cases correctly detected)", fontsize=11)
ax.set_title(
    "ROC Curves — One-vs-Rest per Risk Class\n"
    "AUC closer to 1.0 = better discrimination ability",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])

# Shade the area under the High Risk curve to emphasise it
fpr_hr, tpr_hr, _ = roc_curve(y_test_bin[:, 2], y_pred_proba[:, 2])
ax.fill_between(fpr_hr, tpr_hr, alpha=0.08, color="#C62828")

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/06_roc_curves.png")

# ── PLOT 7: Prediction Confidence Distribution ────────────────────────────────
# For each test prediction, we take the maximum probability (confidence).
# A well-calibrated model is confident when it is correct and less confident
# when it is wrong. This plot shows whether that is the case.
print("          Saving Plot 7: Prediction confidence distribution...")

max_proba = y_pred_proba.max(axis=1)    # Maximum probability for each prediction
correct   = (y_pred == y_test)          # Boolean: was the prediction correct?

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(max_proba[correct],  bins=30, alpha=0.7, color="#2E7D32", label="Correct predictions")
ax.hist(max_proba[~correct], bins=30, alpha=0.7, color="#C62828", label="Wrong predictions")
ax.axvline(0.5, color="grey", linestyle="--", linewidth=1.2, label="50% threshold")
ax.set_title(
    "Model Confidence Distribution\n"
    "Ideally: correct predictions cluster at high confidence, wrong ones at low",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("Prediction Confidence (Maximum Softmax Probability)")
ax.set_ylabel("Number of Predictions")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/07_confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/07_confidence_distribution.png")

# ── PLOT 8: Feature Importance (Permutation-based) ────────────────────────────
# Neural networks do not natively provide feature importance scores.
# We use permutation importance: shuffle each feature one at a time and measure
# how much the accuracy drops. A large drop = the feature is important.
# A small drop = the model barely uses that feature.
print("          Saving Plot 8: Permutation feature importance...")

baseline_acc = test_acc
importances  = []

for i, fname in enumerate(FEATURE_NAMES):
    # Shuffle only feature i — break its relationship with the target
    X_permuted = X_test_s.copy()
    np.random.shuffle(X_permuted[:, i])
    perm_loss, perm_acc = model.evaluate(X_permuted, y_test, verbose=0)
    drop = baseline_acc - perm_acc   # How much did accuracy drop?
    importances.append(drop)
    print(f"            Permuting {fname:<15}: accuracy drop = {drop:+.4f}")

# Sort features by importance (largest drop = most important)
sorted_idx = np.argsort(importances)[::-1]
sorted_features = [FEATURE_NAMES[i] for i in sorted_idx]
sorted_importance = [importances[i] for i in sorted_idx]

# Colour bars: positive drop (important feature) = blue, no drop = grey
bar_colors = ["#1565C0" if v > 0.005 else "#B0BEC5" for v in sorted_importance]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(sorted_features, sorted_importance, color=bar_colors, edgecolor="white", height=0.6)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Accuracy Drop When Feature Is Shuffled\n(Larger = more important to the model)")
ax.set_title(
    "Permutation Feature Importance\n"
    "Which clinical vitals does the model rely on most?",
    fontsize=12, fontweight="bold"
)
# Add value labels on bars
for bar, val in zip(bars, sorted_importance):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=9)
ax.grid(True, axis="x", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/08_feature_importance.png")

# ── PLOT 9: Per-Class Probability Calibration ─────────────────────────────────
# A violin plot of the model's predicted probabilities for each class,
# split by the TRUE class. Well-calibrated: when the true class IS X,
# the model's probability for X should be near 1.0.
print("          Saving Plot 9: Probability calibration by true class...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle(
    "Predicted Probability for Each Class, Grouped by True Class\n"
    "A good model shows high probability for the correct class (diagonal boxes should peak near 1.0)",
    fontsize=12, fontweight="bold"
)

for col_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, ["#2E7D32","#F57C00","#C62828"])):
    prob_data = []
    labels_data = []
    for true_cls, true_name in enumerate(CLASS_NAMES):
        mask = (y_test == true_cls)
        prob_data.append(y_pred_proba[mask, col_idx])
        labels_data.append(true_name)

    # Build a dataframe for seaborn
    plot_df = pd.DataFrame({
        "Probability": np.concatenate(prob_data),
        "True Class":  np.repeat(CLASS_NAMES, [len(p) for p in prob_data])
    })
    sns.boxplot(data=plot_df, x="True Class", y="Probability",
                ax=axes[col_idx], palette=palette_labels, linewidth=1.5)
    axes[col_idx].set_title(f"P(model says '{cls_name}')", fontsize=10, fontweight="bold")
    axes[col_idx].set_ylabel("Predicted Probability" if col_idx == 0 else "")
    axes[col_idx].set_xlabel("")
    axes[col_idx].axhline(0.5, color="grey", linestyle="--", linewidth=1, alpha=0.7)
    axes[col_idx].tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/09_probability_calibration.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"          → Saved: {PLOTS_DIR}/09_probability_calibration.png")




[STEP 8]  Generating evaluation plots...
          Saving Plot 4: Training history (loss + accuracy)...
          → Saved: plots/04_training_history.png
          Saving Plot 5: Confusion matrix...
          → Saved: plots/05_confusion_matrix.png
          Saving Plot 6: ROC curves (one-vs-rest per class)...
          → Saved: plots/06_roc_curves.png
          Saving Plot 7: Prediction confidence distribution...
          → Saved: plots/07_confidence_distribution.png
          Saving Plot 8: Permutation feature importance...
            Permuting Age            : accuracy drop = +0.0083
            Permuting SystolicBP     : accuracy drop = +0.0542
            Permuting DiastolicBP    : accuracy drop = +0.0458
            Permuting BloodGlucose   : accuracy drop = +0.0500
            Permuting BodyTemp       : accuracy drop = +0.0917
            Permuting HeartRate      : accuracy drop = +0.0125
          → Saved: plots/08_feature_importance.png
          Saving Plot 9: Probability ca

In [ ]:


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 9 — CONVERT TO TENSORFLOW LITE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# TensorFlow Lite (TFLite) is a compressed, optimised version of the model
# designed to run on resource-constrained devices (smartphones, microcontrollers).
#
# The conversion process does two things:
# 1. Converts the computation graph to a flat buffer format (.tflite)
# 2. Quantisation (tf.lite.Optimize.DEFAULT) reduces weight precision from
#    32-bit floats to 8-bit integers where possible, shrinking the model
#    size by ~4× with minimal accuracy loss.
#
# The resulting .tflite file is what gets placed in the Android app's
# assets/ folder and loaded at runtime via the TFLite Interpreter API.

print("\n[STEP 9]  Converting model to TensorFlow Lite format...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Apply post-training quantisation:
# Reduces model size and speeds up inference on ARM chips (like those in
# Android phones) that have hardware acceleration for integer arithmetic.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = "maternal_health.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

original_size_kb = sum(p.numpy().nbytes for p in model.trainable_weights) / 1024
tflite_size_kb   = len(tflite_model) / 1024

print(f"\n          Keras model   (float32) : ~{original_size_kb:.1f} KB of weights")
print(f"          TFLite model (quantised) : {tflite_size_kb:.1f} KB  ← file deployed to Android")
print(f"          Size reduction           : {(1 - tflite_size_kb/original_size_kb)*100:.0f}%")
print(f"\n          TFLite model saved → {tflite_path}")

# ── Verify TFLite model gives same predictions ──────────────────────────────
# We run a quick sanity check: feed a sample patient into the TFLite
# interpreter and confirm its output matches the Keras model.

print("\n          Verifying TFLite model output matches Keras model...")

interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_idx = interp.get_input_details()[0]["index"]
out_idx = interp.get_output_details()[0]["index"]

# Test with first 5 samples from the test set
mismatches = 0
for idx in range(5):
    sample = X_test_s[idx:idx+1]                       # Shape: (1, 6)
    keras_pred = np.argmax(model.predict(sample, verbose=0))

    interp.set_tensor(inp_idx, sample)
    interp.invoke()
    tflite_pred = np.argmax(interp.get_tensor(out_idx))

    match = "✓" if keras_pred == tflite_pred else "✗"
    print(f"            Sample {idx+1}: Keras={CLASS_NAMES[keras_pred]:<16}  TFLite={CLASS_NAMES[tflite_pred]:<16}  {match}")
    if keras_pred != tflite_pred:
        mismatches += 1

if mismatches == 0:
    print("\n          TFLite model verified — outputs match Keras model exactly.")
else:
    print(f"\n          WARNING: {mismatches} mismatches detected. Check quantisation settings.")


[STEP 9]  Converting model to TensorFlow Lite format...
Saved artifact at '/tmp/tmpb0c60_2e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='vitals_input')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  137231161355600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161356752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161358672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161359056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161354640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161358288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161358096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161359440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161358480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137231161357328: TensorSpec(shape=(),

In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 10 — LIVE DEMO PREDICTIONS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# This section demonstrates exactly how the Android app uses the model.
# We construct sample patients, normalise their vitals using the saved
# scaler parameters, run inference, and display the result.

print("\n[STEP 10] Live demo predictions on example patients...")
print("          (Reproduces the exact inference pipeline used in the Android app)")

# Three fictional patients covering all three risk profiles
demo_patients = [
    {
        "name":    "Patient A — Aisha, 22 years",
        "vitals":  [22, 110, 72, 6.8, 36.9, 68],
        "note":    "Young mother, normal vitals — should be Low Risk"
    },
    {
        "name":    "Patient B — Nakato, 34 years",
        "vitals":  [34, 132, 85, 8.9, 37.8, 88],
        "note":    "Borderline BP, slightly elevated glucose — should be Moderate Risk"
    },
    {
        "name":    "Patient C — Tumwine, 46 years",
        "vitals":  [46, 158, 100, 12.5, 39.1, 105],
        "note":    "Hypertensive, hyperglycaemic, febrile — should be High Risk"
    },
]

print()
for patient in demo_patients:
    raw = np.array([patient["vitals"]], dtype=np.float32)   # Shape: (1, 6)

    # Apply the SAME normalisation used during training
    raw_scaled = scaler.transform(raw)

    # Get probability output from Keras model
    probs       = model.predict(raw_scaled, verbose=0)[0]   # Shape: (3,)
    pred_class  = np.argmax(probs)
    confidence  = probs[pred_class] * 100

    print(f"  ┌── {patient['name']}")
    print(f"  │   Note     : {patient['note']}")
    print(f"  │   Vitals   : Age={patient['vitals'][0]}  SBP={patient['vitals'][1]}  DBP={patient['vitals'][2]}"
          f"  Glucose={patient['vitals'][3]}  Temp={patient['vitals'][4]}  HR={patient['vitals'][5]}")
    print(f"  │   Probabilities:")
    for i, (cls, p) in enumerate(zip(CLASS_NAMES, probs)):
        bar = "█" * int(p * 30)
        marker = " ← PREDICTED" if i == pred_class else ""
        print(f"  │     {cls:<16} {p*100:5.1f}%  {bar}{marker}")
    print(f"  └── Prediction: {CLASS_NAMES[pred_class].upper()}  (confidence: {confidence:.1f}%)\n")



[STEP 10] Live demo predictions on example patients...
          (Reproduces the exact inference pipeline used in the Android app)

  ┌── Patient A — Aisha, 22 years
  │   Note     : Young mother, normal vitals — should be Low Risk
  │   Vitals   : Age=22  SBP=110  DBP=72  Glucose=6.8  Temp=36.9  HR=68
  │   Probabilities:
  │     Low Risk         100.0%  █████████████████████████████ ← PREDICTED
  │     Moderate Risk      0.0%  
  │     High Risk          0.0%  
  └── Prediction: LOW RISK  (confidence: 100.0%)

  ┌── Patient B — Nakato, 34 years
  │   Note     : Borderline BP, slightly elevated glucose — should be Moderate Risk
  │   Vitals   : Age=34  SBP=132  DBP=85  Glucose=8.9  Temp=37.8  HR=88
  │   Probabilities:
  │     Low Risk           0.0%  
  │     Moderate Risk    100.0%  █████████████████████████████ ← PREDICTED
  │     High Risk          0.0%  
  └── Prediction: MODERATE RISK  (confidence: 100.0%)

  ┌── Patient C — Tumwine, 46 years
  │   Note     : Hypertensive, hype

In [ ]:


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 11 — FINAL SUMMARY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 72)
print("  TRAINING COMPLETE — SUMMARY")
print("=" * 72)
print(f"""
  Dataset
    Total records       : {len(X):,} (balanced: {N_PER_CLASS} per class)
    Training set        : {len(X_train):,} records
    Validation set      : {len(X_val):,} records
    Test set            : {len(X_test):,} records

  Model Architecture
    Type                : Feedforward Neural Network (MLP)
    Layers              : Input(6) → Dense(32,ReLU) → BN → Dropout(0.3)
                          → Dense(16,ReLU) → Dropout(0.2) → Dense(3,Softmax)
    Total parameters    : {model.count_params():,}

  Training
    Epochs completed    : {epochs_run} / 80 (EarlyStopping triggered)
    Best val accuracy   : {best_val_acc:.4f} ({best_val_acc*100:.1f}%)

  Final Test Performance
    Test accuracy       : {test_acc:.4f} ({test_acc*100:.1f}%)
    Test loss           : {test_loss:.4f}

  Output Files
    maternal_health.tflite  ← Copy into Android app/src/main/assets/
    scaler_params.json      ← Reference values for MaternalRiskPredictor.java
    plots/                  ← 9 diagnostic and evaluation plots

  Plots Generated
    01_feature_distributions.png   — Violin plots per class
    02_correlation_heatmap.png     — Feature correlation matrix
    03_class_balance.png           — Class distribution bar chart
    04_training_history.png        — Loss + accuracy per epoch
    05_confusion_matrix.png        — Correct vs incorrect predictions
    06_roc_curves.png              — ROC + AUC per class
    07_confidence_distribution.png — Model confidence analysis
    08_feature_importance.png      — Permutation feature importance
    09_probability_calibration.png — Calibration per true class
""")
print("=" * 72)
print("  Dr. Richard Kimera | Mbarara University of Science and Technology")
print("  MSc Health Informatics — HIN 7102: Applied Health Informatics & AI")
print("=" * 72)

  TRAINING COMPLETE — SUMMARY

  Dataset
    Total records       : 1,200 (balanced: 400 per class)
    Training set        : 840 records
    Validation set      : 120 records
    Test set            : 240 records

  Model Architecture
    Type                : Feedforward Neural Network (MLP)
    Layers              : Input(6) → Dense(32,ReLU) → BN → Dropout(0.3)
                          → Dense(16,ReLU) → Dropout(0.2) → Dense(3,Softmax)
    Total parameters    : 931

  Training
    Epochs completed    : 80 / 80 (EarlyStopping triggered)
    Best val accuracy   : 1.0000 (100.0%)

  Final Test Performance
    Test accuracy       : 1.0000 (100.0%)
    Test loss           : 0.0196

  Output Files
    maternal_health.tflite  ← Copy into Android app/src/main/assets/
    scaler_params.json      ← Reference values for MaternalRiskPredictor.java
    plots/                  ← 9 diagnostic and evaluation plots

  Plots Generated
    01_feature_distributions.png   — Violin plots per class
    02